# Day 3：RoPE 旋转位置编码 —— 位置如何进入 Q/K

> 学习路线定位：  
> **Day 1** 你已经理解了 `QKᵀ / √dₖ / softmax / V`。  
> **Day 2** 你已经理解了 MHA / MQA / GQA 主要影响 `KV Cache` 的显存与带宽。  
> **Day 3** 的目标是回答一个更底层的问题：  
>
> **Transformer 本身是置换不敏感的，它到底怎么知道 token 的顺序？**

---

## 今日学习大纲

| 模块 | 你要学会什么 | 产出 |
|---|---|---|
| 1. 为什么需要位置编码 | 理解 Attention 只看 token 相似度，不天然知道顺序 | 能解释“猫吃鱼”和“鱼吃猫”为什么不能只靠词向量 |
| 2. 绝对位置编码的问题 | 理解 `token_embedding + position_embedding` 的局限 | 能说出为什么长上下文外推容易出问题 |
| 3. RoPE 的核心思想 | 用二维旋转把位置注入 Q/K | 能手写二维旋转矩阵 |
| 4. 相对位置性质 | 推导 `R_mᵀ R_n = R_{n-m}` | 能解释为什么 RoPE 天然表达相对距离 |
| 5. NumPy 复现 | 实现 `apply_rope()`，画旋转图和 attention heatmap | 能运行最小 RoPE demo |
| 6. 工程排障 | 理解 `position_ids`、`rope_theta`、`rope_scaling` 错配问题 | 能定位长上下文质量突然下降的原因 |
| 7. RAG 映射 | 把 RoPE 和 chunk size、overlap、context window 联系起来 | 能解释为什么 RAG 不能盲目塞超长上下文 |

---

## 今日完成目标

完成本 Notebook 后，你应该能够：

1. 画出 RoPE 对 Q/K 的维度变换流程。
2. 写出二维旋转矩阵和 RoPE 的分维度公式。
3. 用 NumPy 验证 RoPE 的两个性质：  
   - 向量范数不变；
   - attention score 依赖相对位置差。
4. 解释为什么 RoPE 作用在 **Q/K**，通常不作用在 **V**。
5. 说清楚扩展 context length 后，为什么模型可能因为 RoPE 配置错误而退化。
6. 把 RoPE 映射到 RAG 的 chunk size、overlap 和 long-context 设计问题。

## 0. 今日环境检查

本 notebook 只需要：

```bash
numpy
pandas
matplotlib
```

如果你已经按 Day 1 安装过：

```bash
conda install -y python=3.11 numpy pandas matplotlib ipykernel jupyterlab
```

通常就可以直接运行。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sin, cos, pi

# 尽量适配 macOS / Windows / Linux 中文字体；如果没有中文字体，图标题可能回退为方框，但不影响代码运行。
plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS", "PingFang SC", "Heiti TC", "Noto Sans CJK SC",
    "SimHei", "Microsoft YaHei", "DejaVu Sans"
]
plt.rcParams["axes.unicode_minus"] = False

np.set_printoptions(precision=4, suppress=True)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

# 1. 为什么 Transformer 需要位置编码？

Self-Attention 的核心计算是：

$$
\text{Attention}(Q,K,V)=\text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

注意这里的 `QKᵀ` 只是在比较 token 表示之间的匹配程度。

如果没有任何位置信息，模型看到的更像是一个“词袋”：

```text
猫 吃 鱼
鱼 吃 猫
```

这两个句子里的 token 集合相同，但语义完全不同。

所以 Transformer 必须额外知道：

```text
第 0 个位置是谁
第 1 个位置是谁
第 2 个位置是谁
两个 token 相隔多远
```

位置编码就是为了解决这个问题。

# 2. 绝对位置编码：简单，但不够优雅

最早常见做法是：

$$
x_m = \text{token\_embedding}_m + \text{position\_embedding}_m
$$

其中：

- `token_embedding` 表示“这个词是什么”
- `position_embedding` 表示“这个词在第几个位置”

这个方法直观，但有几个问题：

| 问题 | 解释 |
|---|---|
| 语义和位置混在一起 | 直接相加后，模型要自己再学会拆分“词义”和“位置” |
| 外推不自然 | 训练只见过 4K 位置，推理突然给 32K，未见过的位置可能不稳定 |
| 相对距离表达不直接 | 很多语言依赖不是“绝对第几个 token”，而是“相隔多远” |

RoPE 的目标不是简单告诉模型“你在第 m 个位置”，而是让 Q/K 的匹配过程天然带上相对位置差。

# 3. RoPE 的第一性原理：二维旋转矩阵

RoPE 的核心动作非常简单：

> 把 Q/K 的相邻两个维度看成一个二维平面，然后按当前位置旋转一个角度。

二维旋转矩阵：

$$
R(\theta)=
\begin{bmatrix}
\cos \theta & -\sin \theta \\
\sin \theta & \cos \theta
\end{bmatrix}
$$

对二维向量：

$$
x =
\begin{bmatrix}
x_1 \\
x_2
\end{bmatrix}
$$

旋转后：

$$
R(\theta)x =
\begin{bmatrix}
x_1\cos\theta - x_2\sin\theta \\
x_1\sin\theta + x_2\cos\theta
\end{bmatrix}
$$

RoPE 对位置 $m$ 的做法是：

$$
x_m^{rot} = R(m\theta)x_m
$$

也就是说：

```text
position = 0 -> 旋转 0θ
position = 1 -> 旋转 1θ
position = 2 -> 旋转 2θ
...
```

不同位置的 token 会被旋转到不同方向。

In [ ]:
def rotation_matrix(theta: float) -> np.ndarray:
    return np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])

# 演示：同一个二维向量在不同位置被旋转
base_vec = np.array([1.0, 0.25])
theta = np.pi / 10
positions = np.arange(0, 12)

rotated = np.array([rotation_matrix(m * theta) @ base_vec for m in positions])

plt.figure(figsize=(7, 7))
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)

for m, v in zip(positions, rotated):
    plt.arrow(0, 0, v[0], v[1], head_width=0.035, length_includes_head=True, alpha=0.75)
    plt.text(v[0] * 1.08, v[1] * 1.08, f"pos {m}", fontsize=9)

circle = plt.Circle((0, 0), np.linalg.norm(base_vec), fill=False, linestyle="--", alpha=0.4)
plt.gca().add_patch(circle)

plt.title("RoPE intuition: same vector rotated by position")
plt.xlabel("dimension 1")
plt.ylabel("dimension 2")
plt.axis("equal")
plt.grid(True, alpha=0.3)
plt.show()

print("原始向量范数:", np.linalg.norm(base_vec))
print("旋转后范数最小值:", np.linalg.norm(rotated, axis=1).min())
print("旋转后范数最大值:", np.linalg.norm(rotated, axis=1).max())

## 你应该观察到什么？

上图里：

- 同一个向量随着 position 增大不断旋转；
- 所有向量长度不变；
- 改变的是方向，不是信息强度。

这很重要：

```text
RoPE 不是把位置当作一个额外向量硬加进去；
RoPE 是把“位置”变成 Q/K 向量方向的一种旋转。
```

# 4. 多维 RoPE：每两个维度一组，每组频率不同

真实 Transformer 的 head_dim 通常不是 2，而是 64、80、96、128 等。

RoPE 把维度两两配对：

```text
(x0, x1), (x2, x3), (x4, x5), ...
```

每一对维度用不同的频率：

$$
\theta_i = \text{base}^{-\frac{2i}{d}}
$$

常见 `base = 10000`，`d = head_dim`。

对于位置 $m$，第 $i$ 对维度的旋转角度是：

$$
m\theta_i
$$

所以第 $i$ 对维度的变换是：

$$
\begin{bmatrix}
x_{2i}^{rot} \\
x_{2i+1}^{rot}
\end{bmatrix}
=
\begin{bmatrix}
\cos(m\theta_i) & -\sin(m\theta_i) \\
\sin(m\theta_i) & \cos(m\theta_i)
\end{bmatrix}
\begin{bmatrix}
x_{2i} \\
x_{2i+1}
\end{bmatrix}
$$

展开为：

$$
x_{2i}^{rot}=x_{2i}\cos(m\theta_i)-x_{2i+1}\sin(m\theta_i)
$$

$$
x_{2i+1}^{rot}=x_{2i}\sin(m\theta_i)+x_{2i+1}\cos(m\theta_i)
$$

不同频率的意义：

| 维度组 | 频率 | 作用直觉 |
|---|---:|---|
| 前面的维度组 | 高频 | 对短距离位置变化敏感 |
| 后面的维度组 | 低频 | 对长距离位置变化更平滑 |

这就是 RoPE 能同时表达短程和长程位置关系的原因之一。

In [ ]:
def rope_angles(seq_len: int, dim: int, base: float = 10000.0) -> np.ndarray:
    # 返回 angles: [seq_len, dim/2]
    # angles[m, i] = m * base^(-2i/dim)
    assert dim % 2 == 0, "RoPE 需要偶数维度，因为要两两配对旋转"
    inv_freq = 1.0 / (base ** (np.arange(0, dim, 2) / dim))
    positions = np.arange(seq_len)
    return np.outer(positions, inv_freq)

def apply_rope(x: np.ndarray, positions=None, base: float = 10000.0) -> np.ndarray:
    # 对 shape [seq_len, dim] 的矩阵应用 RoPE。
    # 这里采用 interleaved pair 风格：(0,1), (2,3), ...
    seq_len, dim = x.shape
    assert dim % 2 == 0, "dim 必须为偶数"

    if positions is None:
        positions = np.arange(seq_len)
    positions = np.asarray(positions)

    inv_freq = 1.0 / (base ** (np.arange(0, dim, 2) / dim))
    angles = np.outer(positions, inv_freq)  # [seq_len, dim/2]
    cos_part = np.cos(angles)
    sin_part = np.sin(angles)

    x_even = x[:, 0::2]
    x_odd = x[:, 1::2]

    out = np.empty_like(x, dtype=float)
    out[:, 0::2] = x_even * cos_part - x_odd * sin_part
    out[:, 1::2] = x_even * sin_part + x_odd * cos_part
    return out

# 测试：范数是否保持不变
rng = np.random.default_rng(42)
x = rng.normal(size=(6, 8))
x_rope = apply_rope(x)

norm_before = np.linalg.norm(x, axis=1)
norm_after = np.linalg.norm(x_rope, axis=1)

pd.DataFrame({
    "position": np.arange(6),
    "norm_before": norm_before,
    "norm_after": norm_after,
    "abs_error": np.abs(norm_before - norm_after)
})

## 结论 1：RoPE 保持向量长度

旋转矩阵是正交矩阵，所以：

$$
\|R(\theta)x\|=\|x\|
$$

这意味着 RoPE 不会因为位置不同而直接放大或缩小向量长度，它主要改变 Q/K 的方向，从而改变 attention score。

# 5. RoPE 最重要的性质：attention score 依赖相对位置

现在看 RoPE 的核心推导。

对 query 向量 $q$ 和 key 向量 $k$，位置分别为 $m$ 和 $n$：

$$
q_m^{rot}=R(m\theta)q
$$

$$
k_n^{rot}=R(n\theta)k
$$

attention score 的点积为：

$$
(q_m^{rot})^\top k_n^{rot}
=
(R(m\theta)q)^\top(R(n\theta)k)
$$

利用旋转矩阵性质：

$$
R(a)^\top R(b)=R(b-a)
$$

得到：

$$
(q_m^{rot})^\top k_n^{rot}
=
q^\top R((n-m)\theta)k
$$

这句话非常关键：

> RoPE 表面上给 Q/K 注入了绝对位置 $m,n$，但进入 dot product 后，分数依赖的是相对位置差 $n-m$。

所以 RoPE 既保留了绝对位置信息，又在 attention score 里自然产生相对位置关系。

In [ ]:
# 验证二维情况下的相对位置性质：
# <R(mθ)q, R(nθ)k> == <q, R((n-m)θ)k>

q = np.array([0.8, -0.4])
k = np.array([0.3, 1.2])
theta = 0.37

rows = []
for m in range(4):
    for n in range(4):
        left = (rotation_matrix(m * theta) @ q).T @ (rotation_matrix(n * theta) @ k)
        right = q.T @ (rotation_matrix((n - m) * theta) @ k)
        rows.append({
            "m": m,
            "n": n,
            "n-m": n - m,
            "left_<Rmq,Rnk>": left,
            "right_<q,R(n-m)k>": right,
            "abs_error": abs(left - right)
        })

pd.DataFrame(rows)

上表里 `abs_error` 应该接近 0。  
这就是 RoPE 相对位置性质的最小数学验证。

# 6. 可视化：同一个相对距离具有相近的 attention pattern

为了让这个性质更直观，我们做一个极简实验：

- 让所有位置的 Q/K 内容向量相同；
- 不使用 RoPE 时，所有位置两两点积都一样；
- 使用 RoPE 后，attention score 会随着相对距离变化；
- 同一条对角线对应同一个 `n-m`，因此会出现对角线结构。

这不是在模拟真实模型，只是用来观察 RoPE 的位置结构。

In [ ]:
seq_len = 24
dim = 8
base_vector = np.zeros(dim)
base_vector[0] = 1.0
base_vector[1] = 0.2
base_vector[2] = 0.6
base_vector[3] = -0.3
base_vector[4] = 0.3
base_vector[5] = 0.7
base_vector[6] = -0.2
base_vector[7] = 0.5

Q_plain = np.tile(base_vector, (seq_len, 1))
K_plain = np.tile(base_vector, (seq_len, 1))

scores_plain = Q_plain @ K_plain.T / np.sqrt(dim)

Q_rope = apply_rope(Q_plain)
K_rope = apply_rope(K_plain)
scores_rope = Q_rope @ K_rope.T / np.sqrt(dim)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im0 = axes[0].imshow(scores_plain, aspect="auto")
axes[0].set_title("Without position: all scores almost identical")
axes[0].set_xlabel("key position n")
axes[0].set_ylabel("query position m")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(scores_rope, aspect="auto")
axes[1].set_title("With RoPE: score depends on relative distance n-m")
axes[1].set_xlabel("key position n")
axes[1].set_ylabel("query position m")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()

## 你应该观察到什么？

左图不含位置信息，所以每个位置互相看起来几乎一样。

右图使用 RoPE 后，出现了明显的条纹 / 对角线结构：

```text
同一条对角线 => n - m 相同 => 相对距离相同
```

这就是为什么 RoPE 能把相对位置差编码进 attention score。

# 7. 频率与周期：为什么 RoPE 能覆盖不同距离？

RoPE 对每一对维度使用不同频率：

$$
\theta_i = 10000^{-\frac{2i}{d}}
$$

对应周期为：

$$
\text{period}_i = \frac{2\pi}{\theta_i}
$$

前面的维度周期短，对局部位置变化敏感；后面的维度周期长，可以覆盖更长距离。

In [ ]:
head_dim = 64
base = 10000.0

i = np.arange(0, head_dim, 2)
theta_i = base ** (-i / head_dim)
period_i = 2 * np.pi / theta_i

freq_df = pd.DataFrame({
    "pair_index": np.arange(len(theta_i)),
    "dimension_pair": [f"({2*j},{2*j+1})" for j in range(len(theta_i))],
    "theta_i": theta_i,
    "period_tokens": period_i
})

display(freq_df.head(8))
display(freq_df.tail(8))

plt.figure(figsize=(9, 5))
plt.plot(freq_df["pair_index"], freq_df["period_tokens"], marker="o")
plt.yscale("log")
plt.title("RoPE periods across dimension pairs")
plt.xlabel("dimension pair index")
plt.ylabel("period in tokens (log scale)")
plt.grid(True, alpha=0.3)
plt.show()

## 解读

- `pair_index` 越小，频率越高，周期越短；
- `pair_index` 越大，频率越低，周期越长；
- 多组频率组合起来，就像不同尺度的“位置尺子”。

这和 Fourier features / 正弦位置编码有相似直觉，但 RoPE 的位置不是直接加到 token embedding 上，而是通过旋转进入 Q/K 的 dot product。

# 8. 为什么 RoPE 通常只作用在 Q 和 K 上，不作用在 V 上？

Attention 的两个阶段：

```text
1. QKᵀ 决定“我该关注谁”
2. softmax 权重 × V 决定“把什么内容汇聚过来”
```

位置关系应该主要影响第 1 步：

$$
\text{score}_{m,n}=q_m^\top k_n
$$

也就是“第 m 个 token 要不要关注第 n 个 token”。

而 V 是被汇聚的信息内容。  
如果把位置强行混入 V，可能会污染内容表示。

所以现代 LLM 里常见做法是：

```text
RoPE(q)
RoPE(k)
V 不旋转
```

面试回答可以这样说：

> RoPE 作用在 Q/K 上，是为了让 attention score 具备相对位置感知；V 是被加权汇聚的内容本身，通常不需要把位置信息直接混入内容向量。

# 9. 常见实现差异：interleaved vs split-half

实际工程里你会看到两类 RoPE 实现：

## A. Interleaved pair

```text
(x0, x1), (x2, x3), (x4, x5), ...
```

## B. Split-half / rotate_half

```text
前半维和后半维配对：
(x0, x_{d/2}), (x1, x_{d/2+1}), ...
```

两种实现只要训练和推理一致即可。  
问题出在：

```text
模型训练时用 A
推理框架却按 B 去转
```

这样会导致 Q/K 的位置结构错位，长上下文质量可能明显下降。

所以排障时要看：

```text
1. 模型原始实现采用哪种 RoPE 排布
2. 推理框架是否兼容该模型
3. HF / llama.cpp / MLX / vLLM 是否使用了正确的 rope implementation
```

In [ ]:
def apply_rope_split_half(x: np.ndarray, positions=None, base: float = 10000.0) -> np.ndarray:
    # split-half 风格：前半和后半维度配对。
    # 这里只用于演示“实现排布差异”，不是说它一定错误。
    seq_len, dim = x.shape
    assert dim % 2 == 0
    half = dim // 2

    if positions is None:
        positions = np.arange(seq_len)
    positions = np.asarray(positions)

    inv_freq = 1.0 / (base ** (np.arange(0, half) / half))
    angles = np.outer(positions, inv_freq)
    cos_part = np.cos(angles)
    sin_part = np.sin(angles)

    x1 = x[:, :half]
    x2 = x[:, half:]

    out = np.empty_like(x, dtype=float)
    out[:, :half] = x1 * cos_part - x2 * sin_part
    out[:, half:] = x1 * sin_part + x2 * cos_part
    return out

rng = np.random.default_rng(0)
x_demo = rng.normal(size=(4, 8))

interleaved = apply_rope(x_demo)
split_half = apply_rope_split_half(x_demo)

print("同一个 x，interleaved RoPE 与 split-half RoPE 的最大差异：")
print(np.abs(interleaved - split_half).max())

print("\n注意：差异本身不代表谁对谁错；关键是训练和推理必须一致。")

# 10. 工程排障 demo：position_ids 重置会发生什么？

长上下文推理或 KV Cache 复用时，一个非常常见的 bug 是：

```text
前 0-5 token position 正确
后 6-11 token position 又从 0 开始
```

这会让模型误以为后半段 token 和前半段 token 处于相同位置区域，attention 的相对位置结构被破坏。

下面用一个 toy demo 观察这个问题。

In [ ]:
seq_len = 12
dim = 8

x_same = np.tile(base_vector, (seq_len, 1))

correct_positions = np.arange(seq_len)
wrong_positions = np.array([0,1,2,3,4,5,0,1,2,3,4,5])  # 模拟 chunk/缓存边界后 position 被错误重置

Q_correct = apply_rope(x_same, positions=correct_positions)
K_correct = apply_rope(x_same, positions=correct_positions)
scores_correct = Q_correct @ K_correct.T / np.sqrt(dim)

Q_wrong = apply_rope(x_same, positions=wrong_positions)
K_wrong = apply_rope(x_same, positions=wrong_positions)
scores_wrong = Q_wrong @ K_wrong.T / np.sqrt(dim)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im0 = axes[0].imshow(scores_correct, aspect="auto")
axes[0].set_title("Correct continuous position_ids: 0..11")
axes[0].set_xlabel("key position")
axes[0].set_ylabel("query position")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(scores_wrong, aspect="auto")
axes[1].set_title("Wrong reset position_ids: 0..5, 0..5")
axes[1].set_xlabel("key position")
axes[1].set_ylabel("query position")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()

print("错误 position_ids:", wrong_positions)

## 解读

右图会出现额外的“重复位置模式”。  
在真实模型中，这种问题可能表现为：

```text
1. 长上下文突然丢失远处信息
2. 模型对后半段内容理解错位
3. 生成重复、跳跃、答非所问
4. prefill 正常，decode 后越来越怪
```

排障关键词：

```text
position_ids
past_key_values
cache_position
rope_theta
rope_scaling
max_position_embeddings
```

# 11. RoPE 与长上下文外推

RoPE 比绝对位置编码更适合长度外推，但不是无限外推。

原因：

- 旋转角度会随着位置增长不断增大；
- 高频维度会周期性重复；
- 训练时没见过很长位置，模型不一定学会在超长距离上稳定使用这些模式；
- 推理框架的 `rope_theta` / `rope_scaling` 配置如果和模型不匹配，会直接破坏位置结构。

常见长上下文技巧包括：

| 技术 | 直觉 |
|---|---|
| RoPE base / theta 调整 | 改变旋转频率，让位置周期变长 |
| NTK scaling | 让模型能更平滑地扩展到更长上下文 |
| YaRN | 更系统地做 RoPE extrapolation |
| Position interpolation | 把长位置压缩映射到训练过的位置范围 |
| Sliding window attention | 不让所有 token 都全局互看，控制成本 |

作为第 3 天，你只需要记住：

> RoPE 让相对位置进入 attention score，但长上下文质量还取决于训练长度、缩放策略和推理框架配置。

# 12. RAG 工程映射：chunk size、overlap、context window

Day 3 的工程映射是：

```text
RoPE / position encoding
    ↓
context window 如何被模型理解
    ↓
RAG 里 chunk size 和 overlap 怎么设计
```

很多人做 RAG 时会犯一个错误：

```text
模型支持 128K context，所以我把所有文档都塞进去。
```

这不等于效果好。

你要考虑：

| 问题 | 解释 |
|---|---|
| 位置衰减 | 很远的上下文不一定被模型稳定使用 |
| 干扰增加 | 无关 chunk 越多，attention 噪声越大 |
| 引用错位 | 模型可能混淆多个相似片段 |
| 成本上升 | prompt token 越多，延迟和成本越高 |
| RoPE 外推风险 | 超出模型稳定训练长度后，位置理解可能退化 |

所以 RAG 通常更需要：

```text
精准 retrieval
合理 chunk size
适度 overlap
rerank
context compression
citation check
```

而不是盲目扩大 context window。

In [ ]:
# 可视化 chunk size / overlap 对文档覆盖的影响

doc_len = 1000
chunk_size = 220
overlap = 60

starts = list(range(0, doc_len, chunk_size - overlap))
chunks = []
for idx, s in enumerate(starts):
    e = min(s + chunk_size, doc_len)
    chunks.append((idx, s, e))
    if e >= doc_len:
        break

plt.figure(figsize=(11, 4))
for idx, s, e in chunks:
    y = idx
    plt.hlines(y, s, e, linewidth=8)
    plt.text((s+e)/2, y+0.12, f"chunk {idx}: {s}-{e}", ha="center", fontsize=9)

plt.title("RAG chunking with overlap: preserving local continuity")
plt.xlabel("document token position")
plt.ylabel("chunk id")
plt.xlim(0, doc_len)
plt.grid(True, axis="x", alpha=0.3)
plt.show()

pd.DataFrame(chunks, columns=["chunk_id", "start_token", "end_token"])

## 解读

`overlap` 的作用不是为了浪费 token，而是为了避免重要语义刚好被切断。

例如：

```text
前一段最后一句提出问题
后一段第一句给出答案
```

如果没有 overlap，retriever 可能只召回其中一半，导致模型答错。

但 overlap 过大也会产生重复上下文，增加干扰和成本。  
这就是 Day 3 和 RAG 的连接点：**模型的位置系统决定了长上下文不是免费午餐。**

# 13. 面试官高频拷问

## Q1：RoPE 为什么通常只作用在 Q/K 上，不作用在 V 上？

答：

Attention 权重由 QKᵀ 决定，位置关系应该影响“我该关注谁”。  
V 是被加权汇聚的信息内容，如果把位置编码强行混入 V，可能污染内容表示。  
RoPE 作用在 Q/K 上，可以让 attention score 具备相对位置感知，而不直接改变被汇聚的 value 内容。

---

## Q2：RoPE 为什么能表达相对位置？

答：

对位置 m 和 n：

$$
(R_mq)^\top(R_nk)=q^\top R_m^\top R_n k=q^\top R_{n-m}k
$$

所以虽然 RoPE 使用的是绝对位置 m、n 的旋转，进入 dot product 后却变成了相对位置差 n-m。

---

## Q3：RoPE 和绝对位置编码有什么区别？

答：

绝对位置编码通常是把 position embedding 加到 token embedding 上：

$$
x_m = token_m + pos_m
$$

RoPE 不直接修改 token embedding，而是旋转 Q/K。  
这样位置关系直接参与 attention score，并天然具有相对位置结构。

---

## Q4：RoPE 是否可以无限支持长上下文？

答：

不可以。  
RoPE 比绝对位置编码更适合外推，但长上下文仍受训练长度、频率周期、rope_theta、rope_scaling、推理框架 position_ids 等影响。  
很多 32K/128K 上下文问题不是模型完全不会，而是位置系统和 cache 系统配置不匹配。

---

## Q5：扩展 context length 后模型质量变差，怎么排查？

答：

优先检查：

```text
1. config.json 的 rope_theta / rope_scaling / max_position_embeddings
2. tokenizer 是否截断
3. prefill 与 decode 的 position_ids 是否连续
4. KV Cache 复用时 cache_position 是否正确
5. 推理框架是否支持该模型的 RoPE 实现细节
```

---

## Q6：RoPE 和 RAG 有什么关系？

答：

RAG 把检索到的 chunks 塞进 context window，而模型通过 RoPE 等位置机制理解这些上下文。  
上下文越长，不代表越好；长距离位置理解、干扰信息、chunk 切分、overlap 和 rerank 都会影响最终质量。  
所以生产级 RAG 依赖精准检索和上下文组织，而不是盲目堆长 prompt。

# 14. 今日最小作业

请你今晚至少完成三件事：

## 作业 A：手写公式

手写或打字整理：

```text
1. 二维旋转矩阵 R(θ)
2. RoPE pair-wise 维度公式
3. (R_m q)^T (R_n k) = q^T R_{n-m} k
```

## 作业 B：改代码参数

修改下面几个值，观察图像变化：

```python
seq_len = 24 -> 64
head_dim = 64 -> 128
base = 10000 -> 500000
chunk_size = 220 -> 400
overlap = 60 -> 100
```

写下你的观察：

```text
base 变大后，周期如何变化？
chunk size 变大后，RAG 上下文风险是什么？
overlap 变大后，召回连续性和重复干扰如何权衡？
```

## 作业 C：面试口述

用 2 分钟说清楚：

```text
RoPE 为什么叫 rotary？
为什么它既用绝对位置，又能表达相对位置？
为什么长上下文出问题时要查 rope_theta 和 position_ids？
```

# 15. 推荐学习资源：只看 3 个高质量视频

你今天只看 3 个就够，不要贪多。

## 1. 3Blue1Brown：Attention in transformers, step-by-step

- 平台：YouTube / 3Blue1Brown 官网
- 用途：补 Day 1 Attention 直觉，适合作为 RoPE 前置复习
- 链接：https://www.3blue1brown.com/lessons/attention

## 2. Rotary Positional Embeddings: Combining Absolute and Relative Position Embeddings

- 平台：YouTube
- 用途：专门讲 RoPE 如何结合绝对位置与相对位置
- 链接：https://www.youtube.com/watch?v=o29P0Kpobz0

## 3. Stanford CS25：Introduction to Transformers w/ Andrej Karpathy

- 平台：YouTube / Stanford CS25
- 用途：把位置编码、Attention、LLM 工程放回完整 Transformer 体系里
- 链接：https://www.youtube.com/watch?v=XfpMkf4rD6E

---

## 辅助阅读

1. RoFormer 原论文：  
   https://arxiv.org/abs/2104.09864

2. Hugging Face RoFormer 文档：  
   https://huggingface.co/docs/transformers/en/model_doc/roformer

3. Hugging Face positional encoding 博客：  
   https://huggingface.co/blog/designing-positional-encoding

# 16. 今日总结

今天你要抓住一句话：

```text
RoPE 不是把位置向量加到 token 上，而是把位置变成 Q/K 的旋转角度；
Q/K 做点积时，绝对位置旋转会转化为相对位置差。
```

这也是为什么 RoPE 是现代 LLM 的核心位置编码方案之一。

明天 Day 4 会进入：

```text
自回归生成
Prefill / Decode
KV Cache 为什么只缓存 K/V 不缓存 Q
上下文长度如何线性放大 KV 显存
```

Day 3 学好后，Day 4 的 KV Cache 位置计数、decode step 和长上下文排障会更容易理解。